In [3]:
#find local directory
getwd()

#set working directory
setwd("/home/user/EDS_Capstone_Project/US_Renewable_Energy_EDS_Capstone/US_Renewable_Energy_EDS_Capstone/Lesson_1:Cleaning_Data_for_Analysis")
getwd()

#set main file path
file_path <- '/home/user/EDS_Capstone_Project/US_Renewable_Energy_EDS_Capstone/US_Renewable_Energy_EDS_Capstone'
file_path

[1] "/home/user/EDS_Capstone_Project/US_Renewable_Energy_EDS_Capstone/US_Renewable_Energy_EDS_Capstone/Lesson_1:Cleaning_Data_for_Analysis"

[1] "/home/user/EDS_Capstone_Project/US_Renewable_Energy_EDS_Capstone/US_Renewable_Energy_EDS_Capstone/Lesson_1:Cleaning_Data_for_Analysis"

[1] "/home/user/EDS_Capstone_Project/US_Renewable_Energy_EDS_Capstone/US_Renewable_Energy_EDS_Capstone"

In [4]:
#load in packages
library(tidyverse)

#load in raw data
governor_data <- read.csv(file.path(file_path, "raw_data/State_Governors_2000_to_2025.csv"))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [5]:
#look at data
#head(governor_data)

#Do not want Gov Name variable
#Need to shorten State to abbreviation
#Need change Time.in.Office -> Year and to split years up

In [6]:
#manipulat data using tidyverse

political_party_data <- governor_data %>% 
    select(-Governor.s.Name	) #delete Gov Name variable

#Abbreviate States to match other datasets
#install.packages("usdata")
library(usdata) #provides vector of all state names so I do not have to type them out

    for (i in seq_along(political_party_data$State)) {
        political_party_data$State[i] = state2abbr(political_party_data$State[i]) #matches state from usdata package and abbreviates it
    }

#change Time.in.Office -> Years to match other datasets
political_party_data$Years <- political_party_data$Time.in.Office
    
#change Party variable -> Political_Party variable
political_party_data$Political_Party <- political_party_data$Party

political_party_data <- political_party_data %>% 
    select(State, Years, Political_Party) #deletes and also changes variable order

#head(political_party_data)

In [7]:
#change time periods into dulicate states with multiple years

#change char - char into char:char
#loops through each row to create char:char
for (x in seq_along(political_party_data$Years)) {
        political_party_data$Years[x] = str_replace(political_party_data$Years[x], " - ", ":")
    }

#loops through each row to create char:char
results <- list()


for (i in seq_along(political_party_data$State)) { #iterated over each governor
    individual_year <- strsplit(political_party_data$Years[i], ":")[[1]] #selects the corresponding time frame
    
    #spits up char:char into num:num
    start_num <- as.numeric(individual_year[1]) 
    end_num <- as.numeric(individual_year[2])

    #this is the range of data/years to iterate over
    yrs <- seq(start_num, end_num)

    #creating the dataframe with the headings I want with the iterated over time frames for each governor
    results[[i]] <- data.frame(
        State = political_party_data$State[i],
        Year = yrs,
        Political_Party = political_party_data$Political_Party[i]
    )
}

political_party_data <- bind_rows(results) #combining all looped governor data

In [8]:
#have to sort down here because running the above code will mess up the kernal

political_party_data <- political_party_data %>%
  arrange(State, Year) %>% #sort in alphabetical order and then numerical order
  distinct() %>%  #deletes duplicate rows = change in politicans, but no change in political party in the same year
  drop_na() #drops rows with NA

head(political_party_data)

#looking for changes in political party in the same state and year
political_party_data <- political_party_data %>%
  group_by(State, Year) %>%
  filter(n() > 1) %>% 
  mutate(Political_Party = paste(Political_Party, collapse = " to ")) %>% #if change in political party = 1st to 2nd
  ungroup() # Added closing parenthesis here

head(political_party_data)

,State,Year,Political_Party
,<chr>,<int>,<chr>
1,AK,1994,Democratic
2,AK,1995,Democratic
3,AK,1996,Democratic
4,AK,1997,Democratic
5,AK,1998,Democratic
6,AK,1999,Democratic


State,Year,Political_Party
<chr>,<int>,<chr>
AK,2002,Republican to Democratic
AK,2002,Republican to Democratic
AK,2014,Independent to Republican
AK,2014,Independent to Republican
AL,2003,Republican to Democratic
AL,2003,Republican to Democratic


In [9]:
write.csv(political_party_data, file.path(file_path, "Lesson_1:Cleaning_Data_for_Analysis/cleaned_datasets/State_Political_Party.csv"))